In [1]:
import numpy as np
import xarray as xr

In [37]:
def parse_dataset(path:str):
    X, Y, wave_number, _counts = np.loadtxt(path, unpack=True)

    X = (X-X.min()).astype(int) 
    Y = (Y-Y.min()).astype(int)

    X_val, X_idx = np.unique(X, return_inverse=True)
    Y_val, Y_idx = np.unique(Y, return_inverse=True)
    wn_val, wn_idx = np.unique(wave_number, return_inverse=True)

    counts = np.zeros((len(X_val), len(Y_val), len(wn_val)), dtype = _counts.dtype)

    for i, j, k, c in zip(X_idx, Y_idx, wn_idx, _counts):
        counts[i, j, k] = c

    counts = xr.DataArray(counts, coords=[X_val, Y_val, wn_val], dims=['X', 'Y', 'wave_number'])
    # Add units attrubutes: X and Y in microns, wave_number in cm^-1
    counts.X.attrs['units'] = 'microns'
    counts.Y.attrs['units'] = 'microns'
    counts.wave_number.attrs['units'] = 'cm^-1'
    return counts

In [2]:
parse_raw = False
data_files = ['../data/Mixed_8x8.txt', '../data/map_c_100x10.txt', '../data/map_c_50x20_mixed2.txt']
system_type = ['SiC+Graphene', 'SiC', 'SiC+Graphene']
if parse_raw:
    datasets = [parse_dataset(path) for path in data_files]
    for i, dataset in enumerate(datasets):
        dataset.attrs['system_type'] = system_type[i]
        #Save to netcdf
        dataset.to_netcdf(f'../data/{system_type[i]}_{len(dataset.X)}x{len(dataset.Y)}.nc')

In [3]:
data_mixed = xr.load_dataarray('../data/SiC+Graphene_50x20.nc')